# LeetCode #1466: Reorder Routes to Make All Paths Lead to the City Zero

https://leetcode.com/problems/reorder-routes-to-make-all-paths-lead-to-the-city-zero/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: BFS/DFS Undirected Traversal ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Try all possible reversals and verify each time whether all paths lead to city 0. Exponential in the worst case.

### Optimal: BFS/DFS Undirected Traversal ★
Treat the tree as **undirected**: for each edge store both the original direction and a flag indicating whether traversal goes "away from 0". BFS from 0: whenever we traverse an edge in its original direction (away from 0 in the rooted sense), that edge must be reversed — count it. Edges traversed opposite to their original direction need no change.

**Constraints:**
* $2 \le n \le 5 \times 10^4$
* Exactly $n - 1$ edges (a tree)

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    public int MinReorder(int n, int[][] connections) {
        // Build undirected adjacency list; tag (neighbour, needsReversal)
        var adj = new List<(int to, bool original)>[n];
        for (int i = 0; i < n; i++) adj[i] = new List<(int, bool)>();
        foreach (var c in connections) {
            adj[c[0]].Add((c[1], true));   // original direction: away from 0
            adj[c[1]].Add((c[0], false));  // reverse direction: toward 0
        }

        // BFS from city 0; count edges that point away (need reversal)
        var visited = new bool[n];
        visited[0] = true;
        var q = new Queue<int>();
        q.Enqueue(0);
        int reversals = 0;

        while (q.Count > 0) {
            int city = q.Dequeue();
            foreach (var (to, original) in adj[city]) {
                if (visited[to]) continue;
                visited[to] = true;
                if (original) reversals++; // edge goes away from 0 — must flip
                q.Enqueue(to);
            }
        }
        return reversals;
    }
}

### Python

In [ ]:
from collections import deque, defaultdict

class Solution:
    def minReorder(self, n: int, connections: list[list[int]]) -> int:
        # Undirected adjacency list; True = original direction (away from 0)
        adj = defaultdict(list)
        for u, v in connections:
            adj[u].append((v, True))   # original: u → v (away from 0 when BFS rooted at 0)
            adj[v].append((u, False))  # reverse: no cost, toward 0

        visited = {0}
        q = deque([0])
        reversals = 0

        while q:
            city = q.popleft()
            for neighbor, is_original in adj[city]:
                if neighbor in visited:
                    continue
                visited.add(neighbor)
                if is_original:
                    reversals += 1  # this edge points away from city 0 — needs reversal
                q.append(neighbor)

        return reversals

### Go

In [ ]:
func minReorder(n int, connections [][]int) int {
	type edge struct{ to int; original bool }
	adj := make([][]edge, n)
	for _, c := range connections {
		adj[c[0]] = append(adj[c[0]], edge{c[1], true})   // original direction
		adj[c[1]] = append(adj[c[1]], edge{c[0], false})  // reverse direction
	}

	visited := make([]bool, n)
	visited[0] = true
	q := []int{0}
	reversals := 0

	for len(q) > 0 {
		city := q[0]; q = q[1:]
		for _, e := range adj[city] {
			if visited[e.to] { continue }
			visited[e.to] = true
			if e.original { reversals++ } // edge points away from 0 — flip it
			q = append(q, e.to)
		}
	}
	return reversals
}

### Rust

In [ ]:
use std::collections::VecDeque;

impl Solution {
    pub fn min_reorder(n: i32, connections: Vec<Vec<i32>>) -> i32 {
        let n = n as usize;
        // (neighbour, is_original_direction)
        let mut adj: Vec<Vec<(usize, bool)>> = vec![vec![]; n];
        for c in &connections {
            adj[c[0] as usize].push((c[1] as usize, true));   // original
            adj[c[1] as usize].push((c[0] as usize, false));  // reverse
        }

        let mut visited = vec![false; n];
        visited[0] = true;
        let mut q: VecDeque<usize> = VecDeque::from([0]);
        let mut reversals = 0;

        while let Some(city) = q.pop_front() {
            for &(to, original) in &adj[city] {
                if visited[to] { continue; }
                visited[to] = true;
                if original { reversals += 1; } // edge points away from city 0
                q.push_back(to);
            }
        }
        reversals
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n=6, connections=[[0,1],[1,3],[2,3],[4,0],[4,5]]`
Edges 0→1, 1→3, 2→3, 4→0, 4→5. BFS from 0 finds: 1 reachable via 0→1 (original, reverse), 4 reachable via 4→0 (reverse, no flip). Processing children of 1: 3 reachable via 1→3 (original, +1). From 4: 5 via 4→5 (original, +1). From 3: 2 via 2→3 reversed (no flip). Answer: **3**.

### 2. Slightly Complex
**Input:** `n=5, connections=[[1,0],[1,2],[3,2],[3,4]]`
Edge 1→0 points toward 0 (no flip). Edge 1→2 points away (+1). Edge 3→2 points toward 0 via reverse (no flip). Edge 3→4 points away (+1). Answer: **2**.

### 3. Edge Case: Time Factor
**Input:** Star graph: all $n-1$ edges point outward from node 0.
BFS from 0 visits all $n-1$ children in one level. No reversal needed (all edges already lead toward 0's children, but since edges go from 0 outward… wait — edges FROM 0 mean neighbours can travel TO 0 directly). Actually edges $0 \to i$ mean $i$ already routes to 0. 0 reversals needed.

### 4. Edge Case: Space Factor
**Input:** Path graph $0-1-2-\cdots-(n-1)$; all edges point away from 0.
Adjacency list holds $2(n-1)$ entries. BFS queue holds at most 1 node at a time on a path. $O(n)$ space total.

### 5. Almost-Impossible but Plausible
**Input:** `n=2, connections=[[1,0]]`
Edge 1→0: BFS from 0 sees neighbour 1 via the reverse (false, no flip). Node 1 can already reach 0. Answer: **0**.